# Manual Implementation of Gaussian Naive Bayes

[Use numpy for all questions (`sklearn` is only allowed to *check* the results)]

In [1]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

## A generative classifier from scratch

Gaussian Naive Bayes models describes what each class *looks like*, and then asks which description fits a new point the best.

*Bayes' rule turns that description into a prediction:*

$$ P(y = k \mid x) \;=\; \frac{P(y = k) \; P(x \mid y = k)}{P(x)} $$

- The prior $P(y = k)$ is how common the class is
- The likelihood $P(x \mid y = k)$ is the model of the class itself
- The denominator does not depend on $k$, so it only normalises
- The whole difficulty sits in $P(x \mid y = k)$:
a joint density over $p$ features is hopeless to estimate from a few hundred rows.
- The **naive** assumption cuts that knot by declaring the features independent **given the class**
$$ P(x \mid y = k) \;=\; \prod_{j=1}^{p} P(x_j \mid y = k) $$

so a $p$-dimensional density becomes $p$ one-dimensional ones.

**Gaussian** naive Bayes then takes each of those to be a normal distribution.

Fitting the model is no more than counting rows and computing a mean and a variance per feature per class:
there is no gradient, no iteration, and no hyperparameter.

Two plotting functions are provided:

In [2]:
def plot_data(X, y):
    plt.plot(X[y == 1, 0], X[y == 1, 1], 'o', color='crimson', ms=4, label='class 1')
    plt.plot(X[y == 0, 0], X[y == 0, 1], 's', color='royalblue', ms=4, label='class 0')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')

In [3]:
def plot_regions(predict, X, y, res=200, pad=1.0, title=None):
    g1 = np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, res)
    g2 = np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, res)
    G1, G2 = np.meshgrid(g1, g2)
    Z = np.asarray(predict(np.c_[G1.ravel(), G2.ravel()])).reshape(G1.shape)
    plt.contourf(G1, G2, Z, levels=[-0.5, 0.5, 1.5],
                 colors=['royalblue', 'crimson'], alpha=0.15)
    plot_data(X, y)
    if title:
        plt.title(title)

Build a dataset made of exactly what the model believes in, two axis-aligned gaussian clouds.
Write a function `blobs(n_per)` returning `X` $(2 n_{per} \times 2)$ and `y`:

- class $1$ is centred on $(1, 1)$, with standard deviations $(1.5, 0.6)$ along the two axes,
- class $0$ is centred on $(-1, 0.5)$, with standard deviations $(0.6, 1.5)$.

Labels are $y_i \in \{0, 1\}$ (integers, they will be used as indices).

Generate a **training set** of $300$ points ($150$ per class) and a **test set** of $1000$ points
($500$ per class), and plot the training set.

*The two clouds are stretched along different axes on purpose; that difference is what will bend the
decision boundary later on.*

### The two ingredients

Everything the model knows about one feature of one class is a mean and a variance. Write the log of the
gaussian density,

$$ \log \mathcal{N}(x \mid \mu, \sigma^2) \;=\; -\frac{1}{2} \log (2 \pi \sigma^2) \;-\; \frac{(x - \mu)^2}{2 \sigma^2} $$

as a function `log_gaussian(x, mu, var)`.

Write it with plain numpy arithmetic, so that it broadcasts: `x` may be a scalar, a vector of $n$ values, or an $(n \times p)$ table, with `mu` and `var` vectors of length $p$.

That single line of broadcasting is what will let you score a whole dataset against a whole class without an explicit loop.

```python
log_gaussian(0.0, 0.0, 1.0),\
log_gaussian(1.0, 0.0, 1.0),\
log_gaussian(3.0, 1.0, 4.0)    # -> -0.9189, -1.4189, -2.1121
```

Now fit the model. Write `fit_gnb(X, y)` returning the four things the model is made of:

- `classes`, the distinct labels, in the order given by `np.unique`,
- `priors`, of shape $(K,)$: the proportion of training rows in each class,
- `means`, of shape $(K \times p)$: the mean of every feature within every class,
- `vars`, of shape $(K \times p)$: the variance of every feature within every class.

Select the rows of one class with a boolean mask, then let `X[y == k].mean(axis=0)` do the work; one loop
over the $K$ classes is enough, and $K$ is $2$ here.

Fit it on the training set and print the four arrays. Do the means and variances match the numbers you used
to generate the data?

*Note that `np.var` divides by $n$, not by $n-1$. That is the maximum-likelihood estimate, and it is what
`sklearn` uses too, so keep it.*

### Scoring a point, in log space

By the naive assumption, the numerator of Bayes' rule for one class is a product, so its logarithm is a sum:

$$ \log \big( P(y = k) \, P(x \mid y = k) \big) \;=\; \log P(y = k) \;+\; \sum_{j=1}^{p} \log \mathcal{N}(x_j \mid \mu_{kj}, \sigma^2_{kj}) $$

Write `log_joint(X, model)`, returning an $(n \times K)$ array holding that quantity for every row and every class.

One loop over the classes, and inside it no loop at all:
`log_gaussian(X, means[k], vars[k])` gives you an $(n \times p)$ array, and summing it over `axis=1` gives the column you want.

Double check you function with this mini test:

```python
X_mini = np.array([[1., 0.], [0., 1.], [2., 3.], [3., 2.]])
y_mini = np.array([0, 0, 1, 1])
model_mini = fit_gnb(X_mini, y_mini)
L = log_joint(X_mini, model_mini)
L # ->[[-2.14472989, -18.14472989], [-2.14472989, -18.14472989], [-18.14472989, -2.14472989], [-18.14472989, -2.14472989]]
```

On the training data, the shape of the output should be $(450, 3)$

Before normalising, see what would have happened without the logarithm.

Stack $100$ copies of the training set's columns side by side (`np.tile(X_train, (1, 100))`, a $200$-feature table), fit the model on it, and look at the smallest entry of `log_joint`.
Compare it with the smallest entry of the normal (untiled) dataset.

What does `np.exp` of these numbers return?
What would that do to a classifier that multiplied $200$ densities together directly?

*Answer:* Probabilities multiplications underflows.

Turn the scores into probabilities.
The missing denominator is $P(x) = \sum_k P(y = k) P(x \mid y = k)$, so

Each term there, numerator and summand alike, is exactly what `log_joint` already computed, read back out of log space:

$$ P(y = k) \, P(x \mid y = k) \;=\; e^{L_k(x)} \qquad \text{with} \qquad L_k(x) \;=\; \log P(y = k) \;+\; \sum_{j=1}^{p} \log \mathcal{N}(x_j \mid \mu_{kj}, \sigma^2_{kj}) $$

Substituting it above and below the bar,

$$ P(y = k \mid x) \;=\; \frac{P(y = k) \, P(x \mid y = k)}{\sum_{k'} P(y = k') \, P(x \mid y = k')} $$

leaves the same quantity on both sides of the fraction, so

$$ P(y = k \mid x) \;=\; \frac{e^{L_k}}{\sum_{k'} e^{L_{k'}}} \qquad \text{where } L_k \text{ is the } k\text{-th column of } \texttt{log\_joint} $$

which is a softmax over the columns.

Written as it stands it overflows or underflows.
The fix is the **log-sum-exp trick**: subtract the row maximum $m_i = \max_k L_{ik}$ from every entry of the row before exponentiating.

The largest exponent becomes $e^0 = 1$, so nothing overflows, and the ratio is unchanged because the factor $e^{-m_i}$ cancels between numerator and denominator.

Write `predict_proba(X, model)` and check that its rows sum to $1$; Then write `predict(X, model)`, which returns `classes[argmax]`.

Print the training and the test accuracy.

Plot the decision regions with `plot_regions`.

### The shape of the boundary

The boundary is the set of points where the two classes tie. In log space that is a difference, and for two
classes it is convenient to look at the **log-odds**

$$ z(x) \;=\; L_1(x) - L_0(x) \;=\; \log \frac{P(y = 1 \mid x)}{P(y = 0 \mid x)} $$

which is positive on one side, negative on the other, and zero on the boundary. Write `log_odds(X, model)`,
and check on a handful of rows that `predict_proba` really is $1 / (1 + e^{-z})$ for class $1$.

Expand $z(x)$ by hand for $p = 2$.

Start from the definition, and note that the sum over the features survives the subtraction:

$$ z(x) \;=\; L_1(x) - L_0(x) \;=\; \underbrace{\log \frac{\pi_1}{\pi_0}}_{\text{priors}} \;+\; \sum_{j=1}^{2} \Big[ \log \mathcal{N}(x_j \mid \mu_{1j}, \sigma^2_{1j}) - \log \mathcal{N}(x_j \mid \mu_{0j}, \sigma^2_{0j}) \Big] $$

Take one feature $j$ and one class $k$, and expand the square in the log-density:

$$ \log \mathcal{N}(x_j \mid \mu_{kj}, \sigma^2_{kj}) \;=\; -\frac{1}{2} \log (2 \pi \sigma^2_{kj}) - \frac{(x_j - \mu_{kj})^2}{2 \sigma^2_{kj}} \;=\; -\frac{x_j^2}{2 \sigma^2_{kj}} \;+\; \frac{\mu_{kj}}{\sigma^2_{kj}} \, x_j \;-\; \frac{\mu_{kj}^2}{2 \sigma^2_{kj}} \;-\; \frac{1}{2} \log (2 \pi \sigma^2_{kj}) $$

which is a quadratic in $x_j$: the first term in $x_j^2$, the second in $x_j$, and the last two do not depend on $x$ at all.

Subtracting the $k = 0$ version from the $k = 1$ version, the $\log 2\pi$ cancels and the three kinds of term collect separately:

$$ \log \mathcal{N}(x_j \mid \mu_{1j}, \sigma^2_{1j}) - \log \mathcal{N}(x_j \mid \mu_{0j}, \sigma^2_{0j}) \;=\; -\frac{1}{2} \Big( \frac{1}{\sigma^2_{1j}} - \frac{1}{\sigma^2_{0j}} \Big) x_j^2 \;+\; \Big( \frac{\mu_{1j}}{\sigma^2_{1j}} - \frac{\mu_{0j}}{\sigma^2_{0j}} \Big) x_j \;+\; c_j $$

$$ \text{with} \qquad c_j \;=\; \frac{\mu_{0j}^2}{2 \sigma^2_{0j}} - \frac{\mu_{1j}^2}{2 \sigma^2_{1j}} + \frac{1}{2} \log \frac{\sigma^2_{0j}}{\sigma^2_{1j}} $$

Summing the two features gives $z$ in full:

$$ z(x) \;=\; a \, x_1^2 \;+\; b \, x_2^2 \;+\; d \, x_1 \;+\; e \, x_2 \;+\; f $$

$$ a = -\frac{1}{2} \Big( \frac{1}{\sigma^2_{11}} - \frac{1}{\sigma^2_{01}} \Big), \qquad b = -\frac{1}{2} \Big( \frac{1}{\sigma^2_{12}} - \frac{1}{\sigma^2_{02}} \Big), \qquad d = \frac{\mu_{11}}{\sigma^2_{11}} - \frac{\mu_{01}}{\sigma^2_{01}}, \qquad e = \frac{\mu_{12}}{\sigma^2_{12}} - \frac{\mu_{02}}{\sigma^2_{02}}, \qquad f = \log \frac{\pi_1}{\pi_0} + c_1 + c_2 $$

The point is what is *missing*. Every term above came from a single feature:
the naive assumption turned the density into a product over $j$, the logarithm turned it into a sum over $j$, and nothing in that sum ever multiplies $x_1$ by $x_2$.

So $z$ is a quadratic function of $x$ with no cross term $x_1 x_2$: it is an axis-aligned conic.

Verify it numerically rather than trusting the algebra:
draw $500$ random points, evaluate `log_odds` on them, and fit the linear system $z \approx a x_1^2 + b x_2^2 + c x_1 x_2 + d x_1 + e x_2 + f$ with `np.linalg.lstsq`.

The residual should be at machine precision, and one of the six coefficients should come out at $0$.

*Hint: the fit is linear in the six coefficients, even though it is quadratic in $x$.*
- Build the $(500 \times 6)$ design matrix by stacking the six columns yourself ($x_1^2$, $x_2^2$, $x_1 x_2$, $x_1$, $x_2$, and a column of ones) with `np.column_stack`, then call `np.linalg.lstsq(A, z, rcond=None)`.
- Draw the points spread over the data, e.g. `np.random.randn(500, 2) * 3`, so the six columns are not collinear.
- Read the residual off `np.abs(A @ coef - z).max()` rather than the second value returned by `lstsq`, which comes back empty in some cases.

Now force the two classes to share the same variances:
replace `vars` by the average of its two rows (`vars.mean(axis=0)`, repeated for both classes) and refit nothing else.

Re-run the same least-squares check with a purely linear model $z \approx d x_1 + e x_2 + f$, and plot the decision regions of this constrained model next to the original ones.

The quadratic cancelled in the subtraction $z = L_1 - L_0$.

In the expansion above the quadratic coefficients were differences of inverse variances,
$$ a = -\frac{1}{2} \Big( \frac{1}{\sigma^2_{11}} - \frac{1}{\sigma^2_{01}} \Big), \qquad b = -\frac{1}{2} \Big( \frac{1}{\sigma^2_{12}} - \frac{1}{\sigma^2_{02}} \Big) $$
so imposing $\sigma^2_{1j} = \sigma^2_{0j} = \sigma^2_j$ makes each bracket a number minus itself, and $a = b = 0$.

Nothing was dropped or approximated: the $x_j^2$ terms are still present in $L_1$ and in $L_0$, with *identical* coefficients, and the subtraction annihilates them.

Feature by feature, before subtracting:

$$ \log \mathcal{N}(x_j \mid \mu_{kj}, \sigma^2_j) \;=\; \underbrace{-\frac{x_j^2}{2 \sigma^2_j}}_{\text{no } k \text{ in it}} \;+\; \frac{\mu_{kj}}{\sigma^2_j} \, x_j \;-\; \frac{\mu_{kj}^2}{2 \sigma^2_j} \;-\; \frac{1}{2} \log (2 \pi \sigma^2_j) $$

Once the variance no longer depends on $k$, neither does the $x_j^2$ term, so it is the same number for both classes and cancels, and the $\tfrac{1}{2}\log(2\pi\sigma^2_j)$ constant with it.

Only the terms that still carry a $k$ survive:

$$ z(x) \;=\; \sum_j \frac{\mu_{1j} - \mu_{0j}}{\sigma^2_j} \, x_j \;+\; \sum_j \frac{\mu_{0j}^2 - \mu_{1j}^2}{2 \sigma^2_j} \;+\; \log \frac{\pi_1}{\pi_0} \;=\; w \cdot x + b $$

*The intuition: $x_j^2$ measures how far out the point sits regardless of direction, which is a statement about spread.*
*When both classes have the same spread, spread carries no evidence about which class the point belongs to, only the location $\mu_{1j} - \mu_{0j}$ does.*
*The quadratic term the part of the score that asks "which class is more tolerant of extreme values".*
*Setting equal variances make that question vacuous.*

Since $z = w \cdot x + b$ and $P(y = 1 \mid x) = \sigma(z)$, the constrained model has the functional form of **logistic regression**, reached by fitting the generative model rather than by maximising the conditional likelihood.

Check your implementation against `sklearn.naive_bayes.GaussianNB`:
compare the training and test accuracies, the fitted `theta_` and `var_` against your `means` and `vars`, and the predicted probabilities.

In [33]:
import sklearn.naive_bayes


-----

## Why *naive* is disserved

The model is fast because it assumes the features carry independent evidence.
When they do not, it does not merely lose accuracy: it stays roughly as accurate and becomes wildly overconfident.

For two classes with equal variances, the log-odds is a sum of one term per feature. 
Each feature adds its own contribution to the confidence.
Hand the model the same measurement twice and it will add that contribution twice, as if a second, independent examination had confirmed the first.

Build the cleanest possible version of that experiment. Write `copies(n, k, noise=0.01)` returning `X`
$(n \times k)$, `y`, and the **true** log-odds of every row:

- draw $y_i$ uniformly in $\{0, 1\}$,
- draw one informative value $x_i \approx \mathcal{N}(+0.5, 1)$ if $y_i = 1$, and $\mathcal{N}(-0.5, 1)$ if $y_i = 0$,
- return $k$ near-identical copies of it, $X_{ij} = x_i + \varepsilon_{ij}$ with $\varepsilon \approx \mathcal{N}(0, \texttt{noise}^2)$.

With equal priors, unit variance and means at $\pm 0.5$, the true log-odds of a row is exactly $x_i$
— derive it from the expansion of the previous section, the quadratic terms cancel and the constants cancel
too. That is the number the model *should* return, whatever $k$ is: the copies bring no new information.

For $k \in \{1, 2, 4\}$:
- Generate a training set of $2000$ rows and a test set of $4000$
- Fit your model
- Plot the fitted log-odds of the test rows against their true log-odds with the line $y = x$ on top.
- Fit the slope of each cloud with `np.polyfit(z_true, z_fitted, 1)` and print it.

Now look at what that does to the probabilities.

For the same three models, plot the histogram of the predicted probability of class $1$ on the test set, and report the fraction of predictions below $0.05$ or
above $0.95$.

-----

## Bernoulli naive Bayes

**The same algorithm, with the gaussian swapped for another density.**

Nothing above used the fact that $P(x_j \mid y = k)$ was a gaussian, only that it was a density we could fit per feature per class.
For binary features the natural choice is a Bernoulli, and the mean and variance are replaced by a single probability $\theta_{kj}$.

Build the data: $p = 10$ binary features, $\theta$ drawn once per class from a `np.random.beta(0.7, 0.7, 10)` so that the two classes differ feature by feature, then $400$ training rows and $2000$ test rows with $x_{ij} \approx \mathrm{Bernoulli}(\theta_{y_i, j})$ and $y_i$ uniform in $\{0, 1\}$.

*Draw the two $\theta$ vectors once and reuse them for both sets, otherwise the test set comes from a different distribution.*

Write `fit_bnb(X, y, alpha=1.0)` and `log_joint_bnb(X, model)`.

**The density.**

Only one line of the algorithm changes: the per-feature density.
A binary feature is described by a single number $\theta_{kj} = P(x_j = 1 \mid y = k)$, and
$$ P(x_j \mid y = k) \;=\; \theta_{kj}^{x_j} \, (1 - \theta_{kj})^{1 - x_j} $$
which reads $\theta_{kj}$ when $x_j = 1$ and $1 - \theta_{kj}$ when $x_j = 0$ (the exponents are switched).

Taking the log and summing over the features,
$$ \log P(x \mid y = k) \;=\; \sum_{j=1}^{p} \Big[ x_j \log \theta_{kj} \;+\; (1 - x_j) \log (1 - \theta_{kj}) \Big] \;=\; x \cdot \log \theta_k \;+\; (1 - x) \cdot \log (1 - \theta_k) $$

The two sums are dot products, so for a whole $(n \times p)$ table they are two matrix products:
`X @ np.log(theta[k])` and `(1 - X) @ np.log(1 - theta[k])`, each of shape $(n,)$.
This is the exact analogue of `log_gaussian(X, means[k], vars[k]).sum(axis=1)`.

**Fitting.**

$\theta_{kj}$ is a proportion: among the rows of class $k$, the fraction that have feature $j$ on.
Counting them is `X[y == k].sum(axis=0)`, a vector of length $p$, and $n_k$ is the number of rows in the class.

Estimate it with **Laplace smoothing**,
$$ \theta_{kj} \;=\; \frac{\big(\sum_{i : y_i = k} x_{ij}\big) + \alpha}{n_k + 2\alpha} $$
which is the plain proportion when $\alpha = 0$:
you add $\alpha$ imaginary rows carrying the feature and $\alpha$ imaginary rows without it, so the estimate can never reach exactly $0$ or $1$.
*(The next question is about what goes wrong when it does.)*

`fit_bnb` mirrors `fit_gnb`, with `theta` replacing the pair `means`/`vars`.
Return:
- `classes`, the distinct labels from `np.unique` / `np.unique_counts`,
- `priors`, of shape $(K,)$: the proportion of training rows in each class,
- `theta`, of shape $(K \times p)$: the smoothed probability of every feature within every class.

One loop over the $K$ classes is enough, as before.

**Scoring.**

`log_joint_bnb(X, model)` is `log_joint` with its one line swapped:
start from an $(n \times K)$ array whose column $k$ is $\log \pi_k$, then add the two matrix products above to column $k$.

The result is the same $(n \times K)$ quantity $L_k(x) = \log P(y = k) + \log P(x \mid y = k)$, so `predict_proba` and `predict` work on it unchanged:
they only ever saw an $(n \times K)$ array of log-joints.

Fit with $\alpha = 1$ and report the test accuracy.

Now the reason $\alpha$ exists:
Wipe out one column for one class in the training set only,
```python
X_train[y_train == 1, 0] = 0    # a symptom that no class-1 patient happened to have
```
and refit with $\alpha = 0$ and with $\alpha = 1$.

Print $\theta_{1,0}$, the test accuracy, and the number of non-finite entries in `log_joint_bnb` in each case.

Check against `sklearn.naive_bayes.BernoulliNB(alpha=1.0)` on the unmodified training set.